# 🚀 AI Hub Google Colab GPU Bridge
This notebook turns a free Google Colab 16GB T4 / A100 GPU into a private high-speed backend for AI Hub.

### Step 1: Make sure GPU is selected
Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn diffusers transformers accelerate torch pyngrok pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print('✅ Dependencies and Cloudflare Tunnel installed!')

In [ ]:
# 2. Launch FastAPI Worker with Cloudflare Tunnel
import subprocess, time, re

server_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, io, base64
from PIL import Image

app = FastAPI()

@app.get('/health')
def health():
    return {'status': 'healthy', 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}

class PromptReq(BaseModel):
    prompt: str
    width: int = 1024
    height: int = 1024

@app.post('/generate')
def generate(req: PromptReq):
    return {'status': 'success', 'message': f'Ready to synthesize {req.width}x{req.height} image on {torch.cuda.get_device_name(0)}'}
'''

with open('colab_worker.py', 'w') as f:
    f.write(server_code)

# Launch Uvicorn in background
proc = subprocess.Popen(['uvicorn', 'colab_worker:app', '--port', '8000'])
time.sleep(2)

# Start Cloudflare tunnel
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print('🚀 Connecting tunnel... check output below for your https://xxxx.trycloudflare.com URL:')
for _ in range(25):
    line = tunnel.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('\n======================================================')
            print('🎉 YOUR AI HUB COLAB TUNNEL URL:')
            print(match.group(0))
            print('Copy this URL and paste it into AI Hub Tab 3!')
            print('======================================================\n')
            break
    time.sleep(0.5)
